# Install all libraries with correct versions

In [1]:
!pip install transformers==4.37.0 accelerate bitsandbytes -q
!pip install chromadb==0.4.24 sentence-transformers -q
!pip install pillow pandas matplotlib -q
!pip install numpy==1.26.4 -q

print("All libraries installed!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.0 which is incompatible.
All libraries installed!


In [2]:
# Check what actually got installed and works
import torch
print("torch:", torch.__version__)

from transformers import Blip2Processor
print("transformers: OK")

import chromadb
print("chromadb:", chromadb.__version__)

from sentence_transformers import SentenceTransformer
print("sentence_transformers: OK")

print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")

torch: 2.11.0+cu128
transformers: OK
chromadb: 0.4.24
sentence_transformers: OK

GPU: Tesla T4


In [5]:
# Extract zip file

import zipfile
import os

with zipfile.ZipFile("/content/CarImage.zip", "r") as z:
    z.extractall("/content/car_data")

print("Extraction done!")

# Show folder structure - only folders with images
for folder, subfolders, files in os.walk("/content/car_data"):
    images = [f for f in files if f.endswith(".jpg") or f.endswith(".png")]
    if images:
        print(f"{folder} → {len(images)} images")

Extraction done!
/content/car_data/test → 8 images
/content/car_data/img → 70 images
/content/car_data/val → 11 images
/content/car_data/train → 59 images


In [6]:
# WHY: BLIP-2 reads images and generates text descriptions.
# 8-bit mode = 4x less GPU memory. Takes 3-5 min first time.

from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

print("Loading BLIP-2... takes 3-5 minutes")

quantization_config = BitsAndBytesConfig(load_in_8bit=True)
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    quantization_config=quantization_config,
    device_map="auto"
)

print("BLIP-2 loaded!")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading BLIP-2... takes 3-5 minutes


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

BLIP-2 loaded!


In [7]:
# WHY: Loop every image → ask BLIP-2 to describe damage →
# collect results into a list → will save as CSV next cell.

import pandas as pd
from PIL import Image

train_folder = "/content/car_data/train"
prompt = "Question: Describe the car damage in detail. Answer:"

image_files = [f for f in os.listdir(train_folder)
               if f.endswith(".jpg") or f.endswith(".jpeg") or f.endswith(".png")]

results = []
print(f"Analysing {len(image_files)} images...\n")

for filename in image_files:
    img_path = os.path.join(train_folder, filename)
    img = Image.open(img_path).convert("RGB")

    inputs = processor(img, text=prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3
    )

    full_response = processor.decode(output[0], skip_special_tokens=True)
    answer = full_response.replace(prompt, "").strip()

    results.append({"image_name": filename, "damage_description": answer})
    print(f"Done: {filename} → {answer[:60]}")

print("\nAll images analysed!")

Analysing 59 images...

Done: 79.jpg → I'm not sure how to describe it, if you mean from where? The
Done: 16.jpg → Toyota Yaris, 1/4 scale
Done: 63.jpg → The front bumper of my 1998 toyota tercel was ripped and the
Done: 25.jpg → A bmw 3 series
Done: 75.jpg → 
Done: 2.jpg → The back of a white suv was smashed with bumpers and rear sp
Done: 68.jpg → The bumper is cracked, and front wheel arch has a dent on it
Done: 23.jpg → 
Done: 31.jpg → -
Done: 18.jpg → The front passenger door of my 2006 chevy volt was damaged, 
Done: 41.jpg → The front windshield was broken by a small branch, and there
Done: 7.jpg → The passenger side door was broken off and hanging down, rea
Done: 50.jpg → there were no cars near by
Done: 13.jpg → I hit a tree on my way home from work, so it was not very im
Done: 64.jpg → 
Done: 43.jpg → The front of my Citroen DS3 was completely destroyed as a re
Done: 56.jpg → The front-end was damaged by a
Done: 54.jpg → it was a very small scratch, but I think that might have 

In [8]:
# WHY: Remove empty/garbage outputs. Save clean version to
# /content so you can download it immediately to your PC.

df = pd.DataFrame(results)

df_clean = df[df["damage_description"].str.len() >= 5].reset_index(drop=True)

print(f"Total: {len(df)} | Clean: {len(df_clean)} | Removed: {len(df) - len(df_clean)}")

df_clean.to_csv("/content/car_damage_clean.csv", index=False)
print("Saved! Right click car_damage_clean.csv in left panel → Download")

df_clean

Total: 59 | Clean: 45 | Removed: 14
Saved! Right click car_damage_clean.csv in left panel → Download


,image_name,damage_description
0,79.jpg,"I'm not sure how to describe it, if you mean f..."
1,16.jpg,"Toyota Yaris, 1/4 scale"
2,63.jpg,The front bumper of my 1998 toyota tercel was ...
3,25.jpg,A bmw 3 series
4,2.jpg,The back of a white suv was smashed with bumpe...
5,68.jpg,"The bumper is cracked, and front wheel arch ha..."
6,18.jpg,The front passenger door of my 2006 chevy volt...
7,41.jpg,The front windshield was broken by a small bra...
8,7.jpg,The passenger side door was broken off and han...
9,50.jpg,there were no cars near by


In [9]:
# WHY: We load our 43 clean descriptions into ChromaDB.
# ChromaDB converts each description into a meaning vector
# and stores it. Later we can search by meaning, not just keywords.

import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd

# Load clean CSV
df_clean = pd.read_csv("/content/car_damage_clean.csv")
print(f"Loaded {len(df_clean)} descriptions")

# Create ChromaDB client and collection
# Collection = like a table that stores text + meaning vectors together
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("car_damage")

# Get descriptions and filenames as plain lists
descriptions = df_clean["damage_description"].tolist()
image_names = df_clean["image_name"].tolist()

# Add all descriptions to ChromaDB
# ids = unique identifier for each row, required by ChromaDB
collection.add(
    documents=descriptions,
    ids=[f"doc_{i}" for i in range(len(descriptions))]
)

print(f"Added {len(descriptions)} descriptions to ChromaDB")
print("Knowledge base ready!")

Loaded 45 descriptions


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 32.2MiB/s]
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Added 45 descriptions to ChromaDB
Knowledge base ready!


In [10]:
# WHY: This is the core of RAG — searching by meaning.
# "rear collision" will match "hit from behind" even though
# the words are completely different. That's the power of vectors.

query = "car bumper damaged after collision"

results = collection.query(
    query_texts=[query],
    n_results=3        # return top 3 most similar descriptions
)

print(f"Query: '{query}'")
print("\nTop 3 similar damage cases found:")
print("-" * 50)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nMatch {i+1}: {doc}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: 'car bumper damaged after collision'

Top 3 similar damage cases found:
--------------------------------------------------

Match 1: The front bumper of my 1998 toyota tercel was ripped and there were a couple dents, both

Match 2: The front bumper is broken and a large dent has been made on it

Match 3: The front bumper of a blue volkswagen beetle is bent and scuffed due to hitting an object


In [11]:
# WHY: This ties everything together into one reusable function.
# Give it any damage query → it finds similar real cases from
# your dataset → returns them as context for answering.

def rag_answer(query):
    print(f"Query: '{query}'")
    print("=" * 50)

    # Step 1: Search ChromaDB for similar descriptions
    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    # Step 2: Extract matched documents
    matched_docs = results["documents"][0]

    # Step 3: Show what was retrieved
    print(f"Found {len(matched_docs)} similar cases:\n")
    for i, doc in enumerate(matched_docs):
        print(f"Case {i+1}: {doc}")

    print("\n" + "=" * 50)
    print("Summary: Based on similar cases in the dataset,")
    print(f"the most relevant damage description is:")
    print(f"\n→ {matched_docs[0]}")

# Test with different queries
rag_answer("scratches on door panel")
print("\n\n")
rag_answer("front end collision damage")
print("\n\n")
rag_answer("rear bumper hit by truck")

Query: 'scratches on door panel'
Found 3 similar cases:

Case 1: it was a very small scratch, but I think that might have been from an edge of my hand or something
Case 2: The front of my vehicle was dented, and some paint is chipped off
Case 3: The front windshield was broken by a small branch, and there were scratches on

Summary: Based on similar cases in the dataset,
the most relevant damage description is:

→ it was a very small scratch, but I think that might have been from an edge of my hand or something



Query: 'front end collision damage'
Found 3 similar cases:

Case 1: The front-end was damaged by a
Case 2: The front part of a silver vehicle has been damaged by an object
Case 3: In my last post I showed a picture of this mercedes benz e320 , it had front end collision and bumper dents

Summary: Based on similar cases in the dataset,
the most relevant damage description is:

→ The front-end was damaged by a



Query: 'rear bumper hit by truck'
Found 3 similar cases:

Case 1: